# Hyperparameter tuning with Optuna
Tuning a random forest's `n_estimators` and `max_depth` on the Pima diabetes data with Optuna's TPE sampler,
then the same study with random and grid samplers, Optuna's plots, and one study that chooses the algorithm too.

The studies are saved in `data/optuna.db`, so the figures of the Note can be drawn from them.

In [ ]:
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message="IProgress not found")   # tqdm's notice about Jupyter widgets
import numpy as np
import optuna
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from optuna.visualization import (plot_contour, plot_optimization_history, plot_parallel_coordinate,
                                  plot_param_importances, plot_slice)

optuna.logging.set_verbosity(optuna.logging.WARNING)    # one line per trial is too much here; set INFO to see them
Path("data/optuna.db").unlink(missing_ok=True)          # start from an empty database every run
STORAGE = "sqlite:///data/optuna.db"

## 1. The data: 768 patients, 8 measurements, Outcome 1 = diabetes

In [ ]:
df = pd.read_csv("data/diabetes.csv")
X = df.drop(columns="Outcome")
y = df["Outcome"]
# 70% to tune and train on, 30% kept for the final test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print(X_train.shape, X_test.shape)

## 2. The objective function: search space + training, returns the mean 3-fold CV accuracy

In [ ]:
def objective(trial):
    # 1. the search space: the sampler picks this trial's values
    n_estimators = trial.suggest_int("n_estimators", 50, 200)
    max_depth = trial.suggest_int("max_depth", 3, 20)
    # 2. the training
    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42, n_jobs=-1)
    return cross_val_score(model, X_train, y_train, cv=3, scoring="accuracy").mean()

## 3. A study with the TPE sampler (Bayesian optimisation), 50 trials

In [ ]:
tpe = optuna.create_study(study_name="tpe", storage=STORAGE, direction="maximize",
                          sampler=optuna.samplers.TPESampler(seed=42))
tpe.optimize(objective, n_trials=50)
print("best CV accuracy:", round(tpe.best_trial.value, 3), " params:", tpe.best_trial.params)

In [ ]:
# the first trials, as Optuna would print them
tpe.trials_dataframe()[["number", "value", "params_n_estimators", "params_max_depth"]].head(8)

## 4. The final model: retrain with the best values on all training rows, test once

In [ ]:
def test_best(study):
    model = RandomForestClassifier(**study.best_trial.params, random_state=42, n_jobs=-1)
    return model.fit(X_train, y_train).score(X_test, y_test)

print("test accuracy:", round(test_best(tpe), 3))

## 5. The same objective with a random sampler and a grid sampler

In [ ]:
rand = optuna.create_study(study_name="random", storage=STORAGE, direction="maximize",
                           sampler=optuna.samplers.RandomSampler(seed=42))
rand.optimize(objective, n_trials=50)

space = {"n_estimators": [50, 100, 150, 200], "max_depth": [5, 10, 15, 20]}
grid = optuna.create_study(study_name="grid", storage=STORAGE, direction="maximize",
                           sampler=optuna.samplers.GridSampler(space, seed=42))
grid.optimize(objective, n_trials=16)        # 4 x 4 combinations

rows = []
for name, s in [("grid", grid), ("random", rand), ("TPE", tpe)]:
    v = np.array([t.value for t in s.trials])
    # how many trials landed in the region that scored best: max_depth 6-10, n_estimators up to 120
    p = s.trials_dataframe()
    region = ((p.params_max_depth.between(6, 10)) & (p.params_n_estimators <= 120)).sum()
    rows.append({"search": name, "trials": len(v), "best CV": v.max(), "mean CV": v.mean(),
                 "trials in region": region, **s.best_trial.params, "test": test_best(s)})
pd.DataFrame(rows).round(3)

## 6. Optuna's plots (Plotly, interactive)

In [ ]:
plot_optimization_history([grid, rand, tpe])

In [ ]:
plot_parallel_coordinate(tpe)

In [ ]:
plot_slice(tpe)

In [ ]:
plot_contour(tpe)

In [ ]:
plot_param_importances(tpe)

## 7. Define-by-run: the algorithm is a hyperparameter too
The search space depends on the first choice: each algorithm suggests only its own hyperparameters.

In [ ]:
def make_model(trial):
    name = trial.suggest_categorical("classifier", ["SVC", "RandomForest", "GradientBoosting"])
    if name == "SVC":
        c = trial.suggest_float("C", 1e-3, 1e3, log=True)
        kernel = trial.suggest_categorical("kernel", ["linear", "rbf"])
        # SVC measures distances, so it gets scaled inputs
        model = make_pipeline(StandardScaler(), SVC(C=c, kernel=kernel, gamma="scale"))
    elif name == "RandomForest":
        model = RandomForestClassifier(n_estimators=trial.suggest_int("n_estimators", 50, 200),
                                       max_depth=trial.suggest_int("max_depth", 3, 20), random_state=42, n_jobs=-1)
    else:
        model = GradientBoostingClassifier(n_estimators=trial.suggest_int("gb_n_estimators", 50, 200),
                                           learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
                                           max_depth=trial.suggest_int("gb_max_depth", 2, 6), random_state=42)
    return model

def objective_any(trial):
    return cross_val_score(make_model(trial), X_train, y_train, cv=3).mean()

anyalgo = optuna.create_study(study_name="algorithm", storage=STORAGE, direction="maximize",
                              sampler=optuna.samplers.TPESampler(seed=42))
anyalgo.optimize(objective_any, n_trials=100)
print("best CV accuracy:", round(anyalgo.best_trial.value, 3), anyalgo.best_trial.params)

In [ ]:
runs = anyalgo.trials_dataframe()
# how often TPE tried each algorithm, and how well it did
runs.groupby("params_classifier")["value"].agg(["count", "mean", "max"]).round(3)

In [ ]:
# when each algorithm was tried: trial numbers in blocks of 20
runs["block"] = runs["number"] // 20 * 20
pd.crosstab(runs["block"], runs["params_classifier"])

In [ ]:
# the best model, rebuilt from the best values: FixedTrial answers every suggest_ call from a dictionary
best_model = make_model(optuna.trial.FixedTrial(anyalgo.best_trial.params))
print(best_model)
print("test accuracy:", round(best_model.fit(X_train, y_train).score(X_test, y_test), 3))